# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze a Croissant-based dataset using the `mlcroissant` library. The dataset contains survey data on mental health indicators among residents of Kilifi County, Kenya, including demographic information and scores for psychological assessments.

### Dataset Source
The dataset is specified via a Croissant schema URL and follows the FAIR² standard:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print("\nDataset publication date:", metadata['datePublished'])
print("Dataset license:", metadata['license'])

## 2. Data Overview
Review available record sets and their properties. All references to entities use their `@id`.

In [ ]:
# List all record sets defined in the dataset
record_sets = dataset.metadata.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")

# Review fields in each RecordSet
for rs in record_sets:
    print(f"\nRecordSet @id: {rs.id} ('{rs.name}') contains fields:")
    for field in rs.fields:
        col_ids = [col.id for col in field.columns]
        print(f"    * Field @id: {field.id}, name: {field.name}, columns @id: {col_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data for analysis from all record sets. Reference using @id.
dataframes = {}

# Store all record set @ids for later reference
record_set_ids = [rs.id for rs in record_sets]

# Load records for each RecordSet
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Choose the first record set (by @id) for demonstration
main_rs_id = record_set_ids[0]
df = dataframes[main_rs_id]
print(f"Columns for RecordSet (@id): {main_rs_id}")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, and grouping. Reference fields and columns by their `@id`.

In [ ]:
# EDA: Filtering and normalizing a numeric field, grouping by category.
rs = [rs for rs in record_sets if rs.id == main_rs_id][0]

# Identify a numeric field for analysis (e.g., GAD-7 total score)
numeric_field = None
for field in rs.fields:
    # Try to find a GAD-7 or PHQ-9 or PSQ score
    if 'gad' in field.name.lower() or 'phq' in field.name.lower() or 'psq' in field.name.lower():
        # Pick the first column @id
        if field.columns:
            numeric_field = field.columns[0].id
            break

if numeric_field is None:
    # Fallback: pick first numeric column
    for field in rs.fields:
        if field.data_type in ['schema:Float', 'schema:Integer', 'schema:Number']:
            if field.columns:
                numeric_field = field.columns[0].id
                break

print("Selected numeric field for analysis (column @id):", numeric_field)

# Filter for values above a threshold (example: 10)
threshold = 10
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head()])

    # Try grouping by a demographic field (e.g., gender, village, etc.)
    group_field = None
    possible_group_fields = ['gender', 'village', 'marital', 'age']
    for field in rs.fields:
        for pf in possible_group_fields:
            if pf in field.name.lower() and field.columns:
                if field.columns[0].id in df.columns:
                    group_field = field.columns[0].id
                    break
        if group_field:
            break

    if group_field:
        print(f"\nGrouping by field (column @id): {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(grouped_df.head())
    else:
        print("No suitable group field found in this record set.")
else:
    print(f"Numeric field {numeric_field} not found in dataframe columns.")

## 5. Visualization
Visualize numeric field distributions and relationships. All fields/columns referenced by their `@id`.

In [ ]:
# Plot histogram for the numeric field
plt.figure(figsize=(8, 5))
if numeric_field in df.columns:
    df[numeric_field].dropna().hist(bins=15)
    plt.title(f"Distribution of {numeric_field} values")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

# If grouping field exists, plot group means
if 'group_field' in locals() and group_field is not None:
    group_means = df.groupby(group_field)[numeric_field].mean().dropna()
    group_means.plot(kind='bar', figsize=(10, 5))
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrates how to load, explore, and process a FAIR² survey dataset using the `mlcroissant` library. By referencing all entities by their `@id`, we ensured consistent access to dataset elements. The exploration included extracting records, filtering on numeric fields (e.g., GAD-7 scores), normalization, and demographic group comparisons. These steps provide a reproducible foundation for deeper mental health analytics and data-driven interventions in Kilifi County, Kenya.

<!-- End of notebook -->